# 00 — Bootstrap do Colab (rodar a cada sessão)

Reconstrói o ambiente: **GPU → Drive → clona → instala (stack moderno) → checa → smoke test → dados**.

> Abra via **File → Open notebook → GitHub** e cole a URL deste repositório.

**Estratégia de deps (revisada):** o Colab roda Python 3.12; versões antigas não
compilam e **não** são necessárias — os frameworks rodam no stack moderno do Colab.
Por isso NÃO fixamos numpy/librosa/etc.

## 1. Configuração — edite só aqui

In [ ]:
# TUDO deriva destas duas linhas. Renomear o projeto = mudar PROJECT_NAME.
GITHUB_USER  = "abUFS"
PROJECT_NAME = "timbre-robust-gtt"

PROJECT_REPO_URL = f"https://github.com/{GITHUB_USER}/{PROJECT_NAME}.git"
PROJECT_DIR      = f"/content/{PROJECT_NAME}"

# Se o repo for PRIVADO, crie um secret GH_TOKEN na aba 🔑 do Colab e
# descomente as 2 linhas abaixo (o clone usará o token):
# from google.colab import userdata
# PROJECT_REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/{GITHUB_USER}/{PROJECT_NAME}.git"

import os
os.environ["TCC_DRIVE_ROOT"] = "/content/drive/MyDrive/tcc-guitar"
os.environ["TCC_LOCAL_ROOT"] = "/content/tcc-data"

FRAMEWORKS = {
    "amt-tools": "https://github.com/cwitkowitz/amt-tools.git",
    "guitar-transcription-with-inhibition":
        "https://github.com/cwitkowitz/guitar-transcription-with-inhibition.git",
    "guitar-transcription-continuous":
        "https://github.com/cwitkowitz/guitar-transcription-continuous.git",  # Fase 4
}
print("Projeto:", PROJECT_DIR)

## 2. Checar GPU  ⚠️ precisa ser T4/GPU, não CPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "SEM GPU! Vá em Runtime > Change runtime type > T4 GPU e reconecte.")
print("GPU OK:", torch.cuda.get_device_name(0))

## 3. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clonar projeto + frameworks

In [ ]:
import os, subprocess

def clone(url, dst):
    if os.path.isdir(dst):
        print(f"[skip] {dst}"); return
    print(f"[clone] {url} -> {dst}")
    subprocess.run(["git", "clone", "--depth", "1", url, dst], check=True)

clone(PROJECT_REPO_URL, PROJECT_DIR)
for name, url in FRAMEWORKS.items():
    clone(url, f"/content/{name}")

os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())

## 5. Instalar (stack moderno — sem pins antigos)

Instala o pouco que falta (`pedalboard`, `PyYAML`) e os frameworks em modo `-e`.
O `amt-tools` e o `inhibition` bastam para a Fase 0 (TabCNN). O `continuous`
(FretNet) é da Fase 4 e depende do `muda` — instalamos com `--no-deps` só para
não travar; ele só será usado lá.

In [ ]:
# 5a. o que falta no Colab
!pip install -q -r {PROJECT_DIR}/requirements-colab.txt

# 5b. frameworks essenciais (Fase 0)
!pip install -q -e /content/amt-tools
!pip install -q -e /content/guitar-transcription-with-inhibition

# 5c. FretNet — adiado (Fase 4); --no-deps evita o muda quebrar o install
!pip install -q -e /content/guitar-transcription-continuous --no-deps || \
    echo "(continuous adiado p/ Fase 4 — ok ignorar)"
print("Instalação concluída.")

> **Restart?** Só é necessário se o pip tiver rebaixado algum pacote já
> importado (raro nesta estratégia). Se o `env_check` acusar erro estranho de
> versão, use *Runtime → Restart session* e rode a partir da célula 1.

## 6. Sanity check (imports + GPU)

In [ ]:
!python -m src.env_check

## 7. Smoke test de runtime (CQT + TabCNN em áudio sintético)

In [ ]:
!python -m src.smoke_test

## 8. Conferir caminhos e criar pastas no Drive

In [ ]:
import sys; sys.path.insert(0, PROJECT_DIR)
from src.paths import P
P.print_summary()
P.ensure_dirs()
print("\npastas garantidas no Drive.")

## 9. Sincronizar dados (Drive → disco local)

Use conforme a fase. Antes da Fase 0 ainda não há cache; depois de processar,
esta célula reconstrói a bancada em segundos.

In [ ]:
from src import data_sync
# data_sync.rsync_raw("guitarset")             # áudio bruto do Drive p/ local
# data_sync.pull_archive("gset_cache", P.local_root)  # cache já processado
print("Ajuste as chamadas conforme a fase.")

---
### Plano B — `condacolab` (Python 3.10) — só se a Fase 4/FretNet exigir

O `muda` (FretNet) não instala no 3.12. Se/quando a Fase 4 travar por isso,
um ambiente conda com Python 3.10 resolve. Para a Fase 0-3 (TabCNN) **não é
preciso** — o stack moderno basta.

```python
!pip install -q condacolab
import condacolab; condacolab.install()   # reinicia o runtime sozinho
```